In [1]:
from pathlib import Path
import sys
import logging

logger = logging.getLogger(__name__)
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
import yaml
import sys 
from src.splitting import (
    create_splits,
    get_label_distribution,
    get_late_ratio,
    get_date_range,
)
from src.data import save_dataframe

In [3]:
import sys 
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)

In [4]:
with open(
    PROJECT_ROOT / "config" / "config.yaml",
    "r",
    encoding="utf-8",
) as file:
    config = yaml.safe_load(file)

In [6]:
input_path = (
    PROJECT_ROOT
    / config["paths"]["labeled_table"]
)

random_state = config["project"]["random_state"]

test_size = config["split"]["test_size"]

validation_size = config["split"]["validation_size"]

logger.info("Input:%s", input_path)
logger.info("Random state:%s", random_state)
logger.info("Test size:%s", test_size)
logger.info("Validation size:%s", validation_size)

2026-09-21 15:16:52,660 - INFO - Input:c:\Users\Anwar Altorkmani\Desktop\MLOps_Project_New\data\processed\labeled_table.csv
2026-09-21 15:16:52,664 - INFO - Random state:42
2026-09-21 15:16:52,668 - INFO - Test size:0.3
2026-09-21 15:16:52,673 - INFO - Validation size:0.5


In [7]:
result = create_splits(
    input_path=input_path,
    test_size=test_size,
    validation_size=validation_size,
    random_state=random_state,
)

In [8]:
train_df = result["train"]
val_df = result["validation"]
test_df = result["test"]

In [9]:
logger.info(
    "Missing labels:%s",
    result["dataset_validation"]["missing_labels"]
)

logger.info(
    "Duplicate orders:%s",
    result["dataset_validation"]["duplicate_orders"]
)

2026-09-21 15:17:11,756 - INFO - Missing labels:0
2026-09-21 15:17:11,759 - INFO - Duplicate orders:0


In [10]:
logger.info("Train:%s", train_df.shape)
logger.info("Validation:%s", val_df.shape)
logger.info("Test:%s", test_df.shape)

2026-09-21 15:17:14,193 - INFO - Train:(67533, 37)
2026-09-21 15:17:14,196 - INFO - Validation:(14471, 37)
2026-09-21 15:17:14,201 - INFO - Test:(14472, 37)


In [11]:
split_validation = result["split_validation"]

logger.info(
    "Total rows:%s",
    split_validation["total_rows"]
)

logger.info(
    "Original rows:%s",
    split_validation["original_rows"]
)

logger.info(
    "Row count matches:%s",
    split_validation["row_count_matches"]
)

2026-09-21 15:17:16,853 - INFO - Total rows:96476
2026-09-21 15:17:16,856 - INFO - Original rows:96476
2026-09-21 15:17:16,860 - INFO - Row count matches:True


In [12]:
for data, name in [
    (train_df, "Train"),
    (val_df, "Validation"),
    (test_df, "Test"),
]:
    distribution = get_label_distribution(data)

    logger.info(f"\n{name}")
    logger.info("-" * 30)

    logger.info(
        "Rows:%s",
        distribution["rows"]
    )

    logger.info(
        "On Time (0):%s %s",
        distribution["on_time_count"],
        f"({distribution['on_time_percentage']:.2f}%)"
    )

    logger.info(
        "Late (1):%s %s",
        distribution["late_count"],
        f"({distribution['late_percentage']:.2f}%)"
    )

2026-09-21 15:17:23,592 - INFO - 
Train
2026-09-21 15:17:23,607 - INFO - ------------------------------
2026-09-21 15:17:23,607 - INFO - Rows:67533
2026-09-21 15:17:23,607 - INFO - On Time (0):62054 (91.89%)
2026-09-21 15:17:23,607 - INFO - Late (1):5479 (8.11%)
2026-09-21 15:17:23,622 - INFO - 
Validation
2026-09-21 15:17:23,622 - INFO - ------------------------------
2026-09-21 15:17:23,637 - INFO - Rows:14471
2026-09-21 15:17:23,637 - INFO - On Time (0):13297 (91.89%)
2026-09-21 15:17:23,649 - INFO - Late (1):1174 (8.11%)
2026-09-21 15:17:23,649 - INFO - 
Test
2026-09-21 15:17:23,675 - INFO - ------------------------------
2026-09-21 15:17:23,686 - INFO - Rows:14472
2026-09-21 15:17:23,707 - INFO - On Time (0):13298 (91.89%)
2026-09-21 15:17:23,713 - INFO - Late (1):1174 (8.11%)


In [13]:
logger.info(
    "Train late ratio:%s",
    get_late_ratio(train_df)
)

logger.info(
    "Validation late ratio:%s",
    get_late_ratio(val_df)
)

logger.info(
    "Test late ratio:%s",
    get_late_ratio(test_df)
)

2026-09-21 15:17:34,315 - INFO - Train late ratio:0.08113070646942976
2026-09-21 15:17:34,320 - INFO - Validation late ratio:0.08112777278695321
2026-09-21 15:17:34,324 - INFO - Test late ratio:0.08112216694306247


In [14]:
logger.info(
    "Train ∩ Validation:%s",
    split_validation[
        "train_validation_overlap"
    ]
)

logger.info(
    "Train ∩ Test:%s",
    split_validation[
        "train_test_overlap"
    ]
)

logger.info(
    "Validation ∩ Test:%s",
    split_validation[
        "validation_test_overlap"
    ]
)

2026-09-21 15:17:37,708 - INFO - Train ∩ Validation:0
2026-09-21 15:17:37,762 - INFO - Train ∩ Test:0
2026-09-21 15:17:37,765 - INFO - Validation ∩ Test:0


In [15]:
for data, name in [
    (train_df, "Train"),
    (val_df, "Validation"),
    (test_df, "Test"),
]:
    date_range = get_date_range(data)

    logger.info(f"\n{name}")

    logger.info(
        "Min:%s",
        date_range["min"]
    )

    logger.info(
        "Max:%s",
        date_range["max"]
    )

2026-09-21 15:17:42,034 - INFO - 
Train
2026-09-21 15:17:42,034 - INFO - Min:2016-10-03 22:31:31
2026-09-21 15:17:42,261 - INFO - Max:2018-08-29 15:00:37
2026-09-21 15:17:42,275 - INFO - 
Validation
2026-09-21 15:17:42,275 - INFO - Min:2016-09-15 12:16:38
2026-09-21 15:17:42,275 - INFO - Max:2018-08-29 14:52:00
2026-09-21 15:17:42,287 - INFO - 
Test
2026-09-21 15:17:42,287 - INFO - Min:2016-10-03 16:56:50
2026-09-21 15:17:42,287 - INFO - Max:2018-08-29 14:18:23


In [17]:
train_path = (
    PROJECT_ROOT
    / config["paths"]["train"]
)

validation_path = (
    PROJECT_ROOT
    / config["paths"]["validation"]
)

test_path = (
    PROJECT_ROOT
    / config["paths"]["test"]
)

In [18]:
save_dataframe(
    train_df,
    train_path,
)

save_dataframe(
    val_df,
    validation_path,
)

save_dataframe(
    test_df,
    test_path,
)

logger.info("Files saved successfully.")

2026-09-21 15:18:37,052 - INFO - Files saved successfully.
